Phase 1 — Foundations and the training pipeline
Start here
Policies, rewards, states, actions, trajectories, token probabilities, and why ordinary SFT is different from RLHF.




Phase 2 — Preference datasets
Build chosen/rejected response pairs, inspect data quality, tokenize conversations, and create a training dataset.




Phase 3 — Reward models
Train a reward model, understand scalar reward scores, pairwise ranking loss, and reward accuracy.




Phase 4 — PPO-based RLHF
Generate responses, score them, calculate advantages, apply PPO clipping, use a reference model and KL penalty, and update the policy.




Phase 5 — DPO and preference optimization
Train directly from preference pairs, compare DPO with PPO, and understand why DPO does not require a separately trained reward model during its main training loop.




Phase 6 — GRPO, evaluation, and practical optimization
Group-based advantages, reward functions, training stability, LoRA/QLoRA, experiment tracking, and comparison of all methods.

📘 Direct Preference Optimization (DPO)
DPO bypasses the traditional RLHF pipeline entirely. It doesn't train a separate reward model or run a complex RL loop; instead, it directly optimizes the language model on a static dataset of preference pairs (a prompt with a "chosen" and a "rejected" response).

How it works: The loss function increases the likelihood of the preferred response while decreasing the likelihood of the rejected one, relative to a frozen reference model. This makes it mathematically equivalent to fitting an implicit reward model, but computationally it's just a supervised fine-tuning step.

When & Why to use it:

Use when: You have a fixed, high-quality dataset of human preferences (e.g., A/B test results, user logs) and your goal is to shape subjective qualities like tone, style, helpfulness, or safety refusals.

Why: It is computationally light (no rollout sampling, no reward model, no critic), stable (essentially a supervised loss), and simple to implement. It's the go-to method for cheap alignment on static data.

Limitations: DPO cannot explore beyond the data it's given. If the optimal response is not present in your dataset, DPO will never discover it. This makes it poor for tasks requiring genuine discovery, like solving novel math problems or writing code.

🧠 Proximal Policy Optimization (PPO)
PPO is the classic RL algorithm that powered InstructGPT and early ChatGPT. It is a full online reinforcement learning method that optimizes a policy by sampling responses, scoring them with a reward model, and updating the policy within a "trust region" to prevent destructive updates.

How it works: PPO uses a clipped surrogate objective to constrain how much the policy can change per update. It relies on a separate critic (value) network to estimate the advantage of each action—essentially predicting how much better a specific response is compared to the average.

When & Why to use it:

Use when: You need maximum flexibility for general RLHF with a learned reward model. This is ideal for complex, subjective tasks where "good" is defined by a subtle model, such as high-stakes alignment or nuanced instruction following.

Why: PPO is the gold standard for alignment fidelity. The critic network provides a per-token baseline that allows for precise, fine-grained policy updates, which is particularly useful when the reward signal is dense and nuanced.

Limitations: It is expensive. You need to keep the policy, reference model, reward model, and critic network in memory simultaneously. It is also sensitive to hyperparameters and prone to reward hacking.

⚡ Group Relative Policy Optimization (GRPO)
GRPO is a memory-efficient variant of PPO designed specifically for reasoning tasks where the reward is verifiable (e.g., math, code). It removes the critic network entirely and replaces the value baseline with a group-relative estimate.

How it works: For each prompt, GRPO samples a group of responses (e.g., 8 to 64). It scores each response, then calculates the advantage of each response by comparing it to the mean and standard deviation of the group's scores. This means the baseline is simply "how well did other attempts at this same question do," eliminating the need for a learned value network.

When & Why to use it:

Use when: You have a cheap, verifiable reward—like unit tests for code, exact-match answers for math, or format constraints. This is the engine behind modern reasoning models like DeepSeek-R1.

Why: It delivers PPO-level online learning without the critic tax. Dropping the value network roughly halves peak memory and speeds up training, allowing you to train reasoning models on more modest hardware.

Limitations: It requires a group of samples per prompt to function, and it doesn't work well when quality is purely subjective. It also has known biases (e.g., favoring shorter correct answers) that variants like Dr. GRPO attempt to fix.

💡 Decision Guide
The cleanest way to choose is to ask about your reward signal:

Is your signal static preference pairs? → Use DPO. It's cheap, stable, and perfect for style and tone alignment.

Do you have a learnable, nuanced reward model and deep pockets? → Use PPO. It offers the most precise control for general RLHF.

Can you verify correctness with a rule or function (math/code)? → Use GRPO. It gives you online RL efficiency without the critic's memory overhead.

In [ ]:
import sys
import torch
import platform

print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())

    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))
        print(
            "Memory (GB):",
            round(torch.cuda.get_device_properties(i).total_memory / 1024**3, 2)
        )

Python: 3.13.5 (tags/v3.13.5:6cb20a2, Jun 11 2025, 16:15:46) [MSC v.1943 64 bit (AMD64)]
Platform: Windows-11-10.0.26200-SP0
PyTorch: 2.6.0+cu124
CUDA available: True
GPU count: 2
GPU 0: NVIDIA GeForce RTX 4060
Memory (GB): 8.0
GPU 1: NVIDIA GeForce RTX 4060
Memory (GB): 8.0


Imagine you are building a chatbot. Users provide you with 10,000 conversations, each labeled to indicate whether a specific response is good or bad. 
Using DPO, you train the model to understand what users prefer. Consequently, the model automatically begins to provide better responses.

In [2]:
from importlib.metadata import version, PackageNotFoundError

packages = [
    "torch",
    "transformers",
    "datasets",
    "accelerate",
    "trl",
    "peft",
    "bitsandbytes",
]

for package in packages:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

torch: 2.6.0+cu124
transformers: 5.19.0
datasets: 5.1.0
accelerate: 1.15.0
trl: 1.14.2
peft: 0.21.2
bitsandbytes: 0.50.2


In [3]:
MODEL_ID = "HuggingFaceTB/SmolLM-135M-Instruct"

SEED = 42
MAX_PROMPT_LENGTH = 128
MAX_RESPONSE_LENGTH = 96

OUTPUT_DIR = "./rlhf_outputs"

print("Model:", MODEL_ID)
print("Seed:", SEED)
print("Output directory:", OUTPUT_DIR)

Model: HuggingFaceTB/SmolLM-135M-Instruct
Seed: 42
Output directory: ./rlhf_outputs


In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Tokenizer class:", tokenizer.__class__.__name__)
print("Vocabulary size:", len(tokenizer))
print("EOS token:", repr(tokenizer.eos_token))
print("PAD token:", repr(tokenizer.pad_token))
print("Chat template available:", tokenizer.chat_template is not None)

messages = [
    {"role": "user", "content": "Explain reinforcement learning in two sentences."}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

print("\nFormatted prompt:\n")
print(prompt)

print("\nPrompt token count:", len(tokenizer.encode(prompt)))

c:\Users\SiddheshLihe\Transformer-LLM-Training-Tuning-Reasoning\venv_train\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokenizer class: GPT2Tokenizer
Vocabulary size: 49152
EOS token: '<|im_end|>'
PAD token: '<|im_end|>'
Chat template available: True

Formatted prompt:

<|im_start|>user
Explain reinforcement learning in two sentences.<|im_end|>
<|im_start|>assistant


Prompt token count: 16


In [5]:

INLINE_DATA = [
    {
        "prompt": "Explain what machine learning is in simple words.",
        "chosen": "Machine learning is a way to teach computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every case.",
        "rejected": "Machine learning is a machine that automatically learns everything without data.",
    },
    {
        "prompt": "What is overfitting in machine learning?",
        "chosen": "Overfitting happens when a model memorizes the training data too closely and performs poorly on new unseen data.",
        "rejected": "Overfitting means the model becomes perfect for all types of data.",
    },
    {
        "prompt": "What is the difference between classification and regression?",
        "chosen": "Classification predicts categories or labels, while regression predicts continuous numerical values.",
        "rejected": "Classification and regression are exactly the same thing.",
    },
    {
        "prompt": "What is the purpose of a loss function?",
        "chosen": "A loss function measures how far a model's predictions are from the true values, guiding the model during training.",
        "rejected": "A loss function increases model size and does not affect training.",
    },
]


In [6]:

required_fields = {"prompt", "chosen", "rejected"}

for i, item in enumerate(INLINE_DATA):
    assert required_fields.issubset(item), (
        f"Example {i} is missing required fields"
    )

    assert all(
        isinstance(item[field], str) and item[field].strip()
        for field in required_fields
    ), f"Example {i} contains an empty or invalid field"

    assert item["chosen"].strip() != item["rejected"].strip(), (
        f"Example {i} has identical chosen/rejected responses"
    )

print("All preference pairs passed validation.")


All preference pairs passed validation.


In [7]:

def inspect_preference_pair(item, tokenizer):
    user_message = {"role": "user", "content": item["prompt"]}

    prompt_text = tokenizer.apply_chat_template(
        [user_message],
        tokenize=False,
        add_generation_prompt=True,
    )

    chosen_text = tokenizer.apply_chat_template(
        [
            user_message,
            {"role": "assistant", "content": item["chosen"]},
        ],
        tokenize=False,
        add_generation_prompt=False,
    )

    rejected_text = tokenizer.apply_chat_template(
        [
            user_message,
            {"role": "assistant", "content": item["rejected"]},
        ],
        tokenize=False,
        add_generation_prompt=False,
    )

    # The formatted text already contains its chat special tokens.
    prompt_ids = tokenizer.encode(
        prompt_text, add_special_tokens=False
    )
    chosen_ids = tokenizer.encode(
        chosen_text, add_special_tokens=False
    )
    rejected_ids = tokenizer.encode(
        rejected_text, add_special_tokens=False
    )

    return {
        "prompt_text": prompt_text,
        "prompt_tokens": len(prompt_ids),
        "chosen_tokens": len(chosen_ids),
        "rejected_tokens": len(rejected_ids),
        "chosen_ids": chosen_ids,
        "rejected_ids": rejected_ids,
    }


example = inspect_preference_pair(INLINE_DATA[0], tokenizer)

print("Formatted prompt:")
print(example["prompt_text"])
print("\nPrompt tokens:", example["prompt_tokens"])
print("Chosen conversation tokens:", example["chosen_tokens"])
print("Rejected conversation tokens:", example["rejected_tokens"])
print("First 15 chosen token IDs:", example["chosen_ids"][:15])

Formatted prompt:
<|im_start|>user
Explain what machine learning is in simple words.<|im_end|>
<|im_start|>assistant


Prompt tokens: 18
Chosen conversation tokens: 46
Rejected conversation tokens: 32
First 15 chosen token IDs: [1, 4093, 198, 36971, 732, 3746, 1380, 314, 281, 2232, 1924, 30, 2, 198, 1]


In [8]:

from datasets import Dataset

preference_dataset = Dataset.from_list(INLINE_DATA)

print(preference_dataset)
print("\nColumns:", preference_dataset.column_names)
print("\nFirst prompt:", preference_dataset[0]["prompt"])
print("\nChosen:", preference_dataset[0]["chosen"])
print("\nRejected:", preference_dataset[0]["rejected"])

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 4
})

Columns: ['prompt', 'chosen', 'rejected']

First prompt: Explain what machine learning is in simple words.

Chosen: Machine learning is a way to teach computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every case.

Rejected: Machine learning is a machine that automatically learns everything without data.


In [9]:
INLINE_DATA.append({
    "prompt": "Explain overfitting in machine learning.",
    "chosen": (
        "Overfitting occurs when a model fits training-specific patterns, "
        "including noise, so its performance drops on unseen data. "
        "Regularization and validation can help detect or reduce it."
    ),
    "rejected": (
        "Overfitting occurs when a model performs well on training data "
        "but poorly on unseen data. It can sometimes be addressed using "
        "regularization."
    ),
})

Init With Model Loading

In [12]:
print(MODEL_ID)

HuggingFaceTB/SmolLM-135M-Instruct


In [38]:
import torch
from transformers import AutoModelForCausalLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

policy_model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
policy_model = policy_model.to(device)
policy_model.eval()

print("Device:", device)
print("Model parameters:", sum(p.numel() for p in policy_model.parameters()))
print("Model loaded successfully.")

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 2806.34it/s]


Device: cuda
Model parameters: 134515008
Model loaded successfully.


In [15]:
import torch
import torch.nn.functional as F

def get_response_logprob(model, tokenizer, prompt, response):
    model.eval()

    messages = [{"role": "user", "content": prompt}]

    # Format the prompt exactly as it would appear before generation.
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # The chat template ends with the assistant prefix.
    # Append the response and the model's EOS token.
    full_text = prompt_text + response + (tokenizer.eos_token or "")

    # Tokenize the completed sequence only once for scoring.
    prompt_ids = tokenizer.encode(
        prompt_text,
        add_special_tokens=False,
    )
    full_ids = tokenizer.encode(
        full_text,
        add_special_tokens=False,
    )

    # Check that the prompt boundary is preserved.
    if full_ids[:len(prompt_ids)] != prompt_ids:
        raise ValueError(
            "Token boundary still differs. Inspect the tokenizer's chat "
            "template and special tokens before scoring."
        )

    input_ids = torch.tensor([full_ids], device=device)
    prompt_length = len(prompt_ids)

    if len(full_ids) <= prompt_length:
        raise ValueError("No response tokens were found.")

    with torch.no_grad():
        logits = model(input_ids=input_ids).logits

    # Position t predicts token t+1.
    log_probs = F.log_softmax(logits[:, :-1, :], dim=-1)
    targets = input_ids[:, 1:]

    token_log_probs = log_probs.gather(
        dim=-1,
        index=targets.unsqueeze(-1),
    ).squeeze(-1)

    # The first response token is at full_ids[prompt_length].
    response_log_probs = token_log_probs[:, prompt_length - 1:]

    return {
        "token_count": response_log_probs.shape[1],
        "sum_logprob": response_log_probs.sum().item(),
        "mean_logprob": response_log_probs.mean().item(),
    }

In [16]:
example = INLINE_DATA[0]

chosen_score = get_response_logprob(
    policy_model,
    tokenizer,
    example["prompt"],
    example["chosen"],
)

rejected_score = get_response_logprob(
    policy_model,
    tokenizer,
    example["prompt"],
    example["rejected"],
)

print("PROMPT:", example["prompt"])

print("\nCHOSEN")
for key, value in chosen_score.items():
    print(f"{key}: {value:.4f}" if isinstance(value, float)
          else f"{key}: {value}")

print("\nREJECTED")
for key, value in rejected_score.items():
    print(f"{key}: {value:.4f}" if isinstance(value, float)
          else f"{key}: {value}")

print("\nDoes the base model favor chosen by summed log-probability?",
      chosen_score["sum_logprob"] > rejected_score["sum_logprob"])

PROMPT: Explain what machine learning is in simple words.

CHOSEN
token_count: 27
sum_logprob: -67.0000
mean_logprob: -2.4844

REJECTED
token_count: 13
sum_logprob: -79.5000
mean_logprob: -6.1250

Does the base model favor chosen by summed log-probability? True


In [18]:
import inspect
import trl
from trl import DPOTrainer, DPOConfig

print("TRL version:", trl.__version__)
print("\nDPOTrainer signature:")
print(inspect.signature(DPOTrainer.__init__))

print("\nDPOConfig signature:")
print(inspect.signature(DPOConfig))

TRL version: 1.14.2

DPOTrainer signature:
(self, model: 'str | PreTrainedModel | PeftModel', ref_model: transformers.modeling_utils.PreTrainedModel | None = None, args: trl.trainer.dpo_config.DPOConfig | None = None, data_collator: collections.abc.Callable[[list[typing.Any]], dict[str, typing.Any]] | None = None, train_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | None = None, eval_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | datasets.dataset_dict.DatasetDict | datasets.dataset_dict.IterableDatasetDict | dict[str, datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset] | None = None, processing_class: transformers.tokenization_utils_base.PreTrainedTokenizerBase | transformers.processing_utils.ProcessorMixin | None = None, compute_metrics: collections.abc.Callable[[transformers.trainer_utils.EvalPrediction], dict] | None = None, callbacks: list[transformers.trainer_callback.TrainerCallb

In [ ]:
preference_dataset

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 4
})

In [20]:
preference_dataset[0]

{'prompt': 'Explain what machine learning is in simple words.',
 'chosen': 'Machine learning is a way to teach computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every case.',
 'rejected': 'Machine learning is a machine that automatically learns everything without data.'}

In [21]:
import gc, torch

del policy_model
gc.collect()
torch.cuda.empty_cache()

print("GPU memory allocated (GB):",
      round(torch.cuda.memory_allocated() / 1024**3, 3))
print("GPU memory reserved  (GB):",
      round(torch.cuda.memory_reserved() / 1024**3, 3))

GPU memory allocated (GB): 0.259
GPU memory reserved  (GB): 0.281


Configure DPO

In [27]:


dpo_config = DPOConfig(
    output_dir=OUTPUT_DIR,

    # ---- core DPO hyperparameters ----
    beta=0.1,                       # KL strength. 0.1 is the canonical default.
                                    # Higher = stay closer to reference. Lower = drift more.

    # ---- training schedule ----
    num_train_epochs=3,
    per_device_train_batch_size=1,  # small because we have 4 rows
    gradient_accumulation_steps=4,  # effective batch = 4
    learning_rate=5e-6,             # DPO is sensitive; 1e-6 to 5e-6 typical
    lr_scheduler_type="cosine",
    warmup_steps=0.1,
    max_grad_norm=1.0,

    # ---- memory ----
    gradient_checkpointing=True,
    bf16=True,                      # 4060 supports bf16
    fp16=False,
    optim="adamw_torch",
    use_cache=False,                # must be False with gradient_checkpointing

    # ---- sequence length ----
    max_length=MAX_PROMPT_LENGTH,                 # plenty for your short examples
    truncation_mode="keep_start",

    # ---- logging / saving ----
    logging_steps=1,
    save_strategy="epoch",
    report_to="none",               # change to "wandb" if you use it
    seed=42,
)

In [28]:
trainer = DPOTrainer(
    model=MODEL_ID,                  # string → TRL loads it
    ref_model=None,                  # implicit frozen reference
    args=dpo_config,
    train_dataset=preference_dataset,
    processing_class=tokenizer,      # note: processing_class, not tokenizer
)

Tokenizing train dataset: 100%|██████████| 4/4 [00:00<00:00, 790.30 examples/s]
Dropping fully truncated examples from train dataset: 100%|██████████| 4/4 [00:00<00:00, 675.66 examples/s]
Loading weights: 100%|██████████| 272/272 [00:00<00:00, 1624.43it/s]


In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
trainer = DPOTrainer(
    model=policy_model, ref_model=None, args=dpo_config,
    train_dataset=preference_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,         # TRL uses the base model as reference automatically
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Tokenizing train dataset: 100%|██████████| 4/4 [00:00<00:00, 1356.94 examples/s]
Dropping fully truncated examples from train dataset: 100%|██████████| 4/4 [00:00<00:00, 3206.04 examples/s]


In [30]:
#sanity
# # Grab one raw example and run it through the collator
sample = preference_dataset[0]
batch = trainer.data_collator([trainer.train_dataset[0]])

for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f"{k:25s} shape={tuple(v.shape)} dtype={v.dtype}")
    else:
        print(f"{k:25s} {type(v)}")

input_ids                 shape=(2, 36) dtype=torch.int64
attention_mask            shape=(2, 36) dtype=torch.int64
completion_mask           shape=(2, 36) dtype=torch.int64


In [31]:
ids = batch["input_ids"][0]
mask = batch["completion_mask"][0] if "completion_mask" in batch else None
print(tokenizer.decode(ids[mask.bool()]) if mask is not None else tokenizer.decode(ids))

Machine learning is a way to teach computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every case.<|im_end|>


In [32]:
train_result = trainer.train()
print(train_result)

Step,Training Loss
1,0.693147
2,0.693147
3,0.688548


TrainOutput(global_step=3, training_loss=0.6916139920552572, metrics={'train_runtime': 6.225, 'train_samples_per_second': 1.928, 'train_steps_per_second': 0.482, 'total_flos': 470651233536.0, 'train_loss': 0.6916139920552572, 'epoch': 3.0})


In [33]:
trainer.save_model("./rlhf_outputs/dpo_run1/final")
tokenizer.save_pretrained("./rlhf_outputs/dpo_run1/final")

('./rlhf_outputs/dpo_run1/final\\tokenizer_config.json',
 './rlhf_outputs/dpo_run1/final\\chat_template.jinja',
 './rlhf_outputs/dpo_run1/final\\tokenizer.json')

In [34]:
trained_model = trainer.model
trained_model.eval()

# Reuse your function, but pass the trained model
for item in INLINE_DATA[:3]:
    c = get_response_logprob(trained_model, tokenizer, item["prompt"], item["chosen"])
    r = get_response_logprob(trained_model, tokenizer, item["prompt"], item["rejected"])
    print(f"Prompt: {item['prompt'][:50]}...")
    print(f"  chosen  mean logp: {c['mean_logprob']:.4f}")
    print(f"  rejected mean logp: {r['mean_logprob']:.4f}")
    print(f"  margin: {c['mean_logprob'] - r['mean_logprob']:.4f}")

Prompt: Explain what machine learning is in simple words....
  chosen  mean logp: -2.4980
  rejected mean logp: -6.1444
  margin: 3.6464
Prompt: What is overfitting in machine learning?...
  chosen  mean logp: -2.7466
  rejected mean logp: -5.8608
  margin: 3.1143
Prompt: What is the difference between classification and ...
  chosen  mean logp: -3.0653
  rejected mean logp: -3.1704
  margin: 0.1051


In [39]:
# Fresh examples the model has NEVER seen
FRESH_EXAMPLES = [
    {
        "prompt": "What is a neural network?",
        "chosen": "A neural network is a model of connected layers that learn patterns from data by adjusting weights.",
        "rejected": "A neural network is a computer that thinks like a human brain.",
    },
    {
        "prompt": "What is gradient descent?",
        "chosen": "Gradient descent is an algorithm that adjusts parameters step by step to reduce the loss.",
        "rejected": "Gradient descent is a method that makes the loss go up.",
    },
    {
        "prompt": "What is a validation set?",
        "chosen": "A validation set is data used to tune settings and catch overfitting before final testing.",
        "rejected": "A validation set is the same as the training set.",
    },
]

def check_margins(model, label, examples):
    print(f"\n===== {label} =====")
    for item in examples:
        c = get_response_logprob(model, tokenizer, item["prompt"], item["chosen"])
        r = get_response_logprob(model, tokenizer, item["prompt"], item["rejected"])
        margin = c["mean_logprob"] - r["mean_logprob"]
        verdict = "✓ chosen wins" if margin > 0 else "✗ REJECTED wins"
        print(f"\nPrompt: {item['prompt']}")
        print(f"  chosen   mean logp: {c['mean_logprob']:.4f}")
        print(f"  rejected mean logp: {r['mean_logprob']:.4f}")
        print(f"  margin: {margin:+.4f}   {verdict}")


# ---- Fresh examples (the honest test) ----
check_margins(policy_model,   "BASE model on FRESH examples",   FRESH_EXAMPLES)
check_margins(trained_model,"DPO model on FRESH examples",    FRESH_EXAMPLES)

# ---- Training examples (to confirm it learned the training set) ----
check_margins(policy_model,   "BASE model on TRAINING examples", INLINE_DATA[:3])
check_margins(trained_model,"DPO model on TRAINING examples",  INLINE_DATA[:3])


===== BASE model on FRESH examples =====

Prompt: What is a neural network?
  chosen   mean logp: -4.0625
  rejected mean logp: -3.4844
  margin: -0.5781   ✗ REJECTED wins

Prompt: What is gradient descent?
  chosen   mean logp: -3.7031
  rejected mean logp: -5.1875
  margin: +1.4844   ✓ chosen wins

Prompt: What is a validation set?
  chosen   mean logp: -4.6875
  rejected mean logp: -3.3281
  margin: -1.3594   ✗ REJECTED wins

===== DPO model on FRESH examples =====

Prompt: What is a neural network?
  chosen   mean logp: -4.0609
  rejected mean logp: -3.4942
  margin: -0.5667   ✗ REJECTED wins

Prompt: What is gradient descent?
  chosen   mean logp: -3.7319
  rejected mean logp: -5.2058
  margin: +1.4740   ✓ chosen wins

Prompt: What is a validation set?
  chosen   mean logp: -4.6897
  rejected mean logp: -3.2944
  margin: -1.3953   ✗ REJECTED wins

===== BASE model on TRAINING examples =====

Prompt: Explain what machine learning is in simple words.
  chosen   mean logp: -2.4844
 

In [40]:
def generate_response(
    prompt: str,
    model,
    tokenizer,
    max_new_tokens: int = 150,
    temperature: float = 0.7,
    top_p: float = 0.9,
) -> str:
    """
    Generate a response from a model given a user prompt.
    
    Args:
        prompt: user's question
        model: the loaded model (base_model or trained_model)
        tokenizer: the tokenizer
        max_new_tokens: how many tokens to generate
        temperature: 0 = greedy (deterministic), >0 = creative
        top_p: nucleus sampling cutoff
    
    Returns:
        generated response as a string
    """
    messages = [{"role": "user", "content": prompt}]

    # Format using the chat template (adds <|im_start|> etc.)
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # Tokenize and move to the model's device
    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
    ).to(model.device)

    prompt_token_count = inputs["input_ids"].shape[1]

    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }

    if temperature > 0:
        generation_kwargs.update({
            "do_sample": True,
            "temperature": temperature,
            "top_p": top_p,
        })
    else:
        generation_kwargs["do_sample"] = False

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            **generation_kwargs,
        )

    new_tokens = outputs[0, prompt_token_count:]
    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()

In [44]:
# Test prompts
test_prompts = [
    # "Explain machine learning in simple words.",
    # "What is overfitting?",
    # "How are classification and regression different?",
    "What does a loss function do?",
    "Why is validation data useful?",
]
count = 0
for p in test_prompts:
    print("=" * 90)
    print("PROMPT:", p)
    print("-" * 90)
    count += 1
    print(f"\nTest {count}: {p}\n")
    base_resp = generate_response(
        p,
        model=policy_model,
        tokenizer=tokenizer,
        temperature=0.3,
        max_new_tokens=120,
    )
    dpo_resp = generate_response(
        p,
        model=trained_model,
        tokenizer=tokenizer,
        temperature=0.3,
        max_new_tokens=120,
    )

    print("\n🔵 BASE model:")
    print(base_resp)
    print("\n🟢 DPO model:")
    print(dpo_resp)
    print()

PROMPT: What does a loss function do?
------------------------------------------------------------------------------------------

Test 1: What does a loss function do?


🔵 BASE model:
A loss function is a mathematical function that measures the difference between the predicted probabilities of a model's output and the true labels. The goal of a loss function is to minimize the difference between the predicted probabilities and the true labels, so that the model can learn to make accurate predictions.

The key idea behind a loss function is to identify the loss function that minimizes the difference between the predicted probabilities and the true labels. This is often referred to as the "loss" or "loss function."

There are several types of loss functions, including:

1. **Mean Squared Error (

🟢 DPO model:
A loss function is a mathematical function that measures the difference between the predicted output and the actual output of a machine learning model. The goal of a loss function i